# Train BiLSTM with MLflow Integration

This notebook trains a deep learning BiLSTM on our ticket dataset, utilizing the custom Word2Vec embeddings we built earlier.

It also connects directly to our existing MLflow server, logging the neural network training curves alongside our classical metrics (Macro F1, Accuracy) so we can do an apples-to-apples comparison against LinearSVC.

In [28]:
import pandas as pd
import numpy as np
import os
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from gensim.models import Word2Vec

# Check for GPU and enable memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n>>> FOUND GPU: {gpus} <<<")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth enabled successfully! TensorFlow will not hoard VRAM.")
    except RuntimeError as e:
        print(e)
else:
    print("\n>>> NO GPU FOUND! Training on CPU. <<<")


>>> NO GPU FOUND! Training on CPU. <<<


In [29]:
# 1. Load Data
data_path = "../artifacts/data_transformation/transformed_dataset.csv"
df = pd.read_csv(data_path)
df = df.dropna(subset=['body', 'queue'])

X = df['body'].astype(str)
y = df['queue']

X_train_text, X_test_text, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training instances: {len(X_train_text)}")
print(f"Test instances: {len(X_test_text)}")

Training instances: 13070
Test instances: 3268


In [31]:
# 2. Tokenize and Pad Sequences
MAX_WORDS = 10000
MAX_LEN = 50

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Padded X_train shape: {X_train.shape}")

Padded X_train shape: (13070, 50)


In [32]:
# 3. Encode Labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
num_classes = len(le.classes_)

print(f"Found {num_classes} classes.")

Found 10 classes.


In [33]:
# 4. Load Custom Word2Vec & Build Embedding Matrix
EMBEDDING_DIM = 100
w2v_model = Word2Vec.load("../artifacts/word2vec/skipgram_v100.model")

word_index = tokenizer.word_index
vocab_size = min(MAX_WORDS, len(word_index) + 1)

embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))
hits, misses = 0, 0

for word, i in word_index.items():
    if i >= vocab_size:
        continue
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]
        hits += 1
    else:
        misses += 1

print(f"Loaded {hits} words from our custom Word2Vec model.")
print(f"Missed {misses} words (mostly rare words or <OOV>).")

Loaded 3171 words from our custom Word2Vec model.
Missed 681 words (mostly rare words or <OOV>).


In [34]:
# 5. Build BiLSTM Architecture
# TensorFlow automatically executes this on the GPU if the check above found one!
model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        trainable=True, # Allow the BiLSTM to fine-tune our Word2Vec embeddings!
        mask_zero=True
    ),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

c:\Users\Ayan\Documents\GitHub\AIML\NLP- customer ticket classfier\.venv\Lib\site-packages\keras\src\layers\layer.py:1039: UserWarning: Layer 'global_max_pooling1d_4' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 50, 100)        │       385,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 50, 128)        │        84,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_4          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 478,686 (1.83 MB)

 Trainable params: 478,686 (1.83 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
# 6. Callbacks & MLflow Configuration
os.makedirs("../artifacts/models", exist_ok=True)
checkpoint_path = "../artifacts/models/bilstm_best.keras"

# Early Stopping: Halt if validation loss stops improving for 6 epochs
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

# Model Checkpoint: Save the best model dynamically
checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True)

# Configure MLflow (use absolute path for sqlite db since we are in a subfolder)
db_path = os.path.abspath("../mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("Customer Support Ticket Classification")

# Turn on TensorFlow Autologging (Captures epochs, batch size, learning rate, and curves!)
mlflow.tensorflow.autolog()

In [36]:
# 7. Train & Log
from sklearn.utils.class_weight import compute_class_weight

print("Starting MLflow Run...")
with mlflow.start_run(run_name="BiLSTM_Word2Vec"):
    
    # Compute class weights to handle imbalanced ticket queues
    weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_enc), y=y_train_enc)
    class_weights_dict = dict(enumerate(weights))
    
    # Train the network
    history = model.fit(
        X_train, y_train_enc,
        validation_split=0.1,
        epochs=25,
        batch_size=32,
        class_weight=class_weights_dict,
        callbacks=[early_stop, checkpoint]
    )
    
    print("\n\n=== Neural Network Training Complete! Evaluation Phase ===\n")
    
    # Predict on the unseen Test set
    y_pred_probs = model.predict(X_test)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    # Calculate classical metrics so they align with our SVM runs in MLflow
    acc = accuracy_score(y_test_enc, y_pred)
    prec = precision_score(y_test_enc, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test_enc, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test_enc, y_pred, average='macro')
    
    # Log the classical metrics explicitly
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("macro_precision", prec)
    mlflow.log_metric("macro_recall", rec)
    mlflow.log_metric("macro_f1", f1)
    
    # Log Text Report
    report = classification_report(y_test_enc, y_pred, target_names=le.classes_)
    with open("classification_report.txt", "w") as f:
        f.write(report)
    mlflow.log_artifact("classification_report.txt", artifact_path="evaluation_metrics")
    
    # Log Confusion Matrix Heatmap
    cm = confusion_matrix(y_test_enc, y_pred)
    plt.figure(figsize=(10,7))
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('BiLSTM Confusion Matrix')
    plt.savefig("confusion_matrix.png")
    plt.close()
    mlflow.log_artifact("confusion_matrix.png", artifact_path="evaluation_metrics")
    
    print(f"\nFinal Test Accuracy: {acc:.4f}")
    print(f"Final Test Macro F1: {f1:.4f}")

print("\nSuccessfully logged everything to MLflow! Go check the UI!")


Starting MLflow Run...


Epoch 1/25


c:\Users\Ayan\Documents\GitHub\AIML\NLP- customer ticket classfier\.venv\Lib\site-packages\keras\src\layers\layer.py:1039: UserWarning: Layer 'global_max_pooling1d_4' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


368/368 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.2102 - loss: 2.0963

368/368 ━━━━━━━━━━━━━━━━━━━━ 27s 56ms/step - accuracy: 0.2102 - loss: 2.0963 - val_accuracy: 0.2288 - val_loss: 1.9825
Epoch 2/25
367/368 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2359 - loss: 1.9240

...

368/368 ━━━━━━━━━━━━━━━━━━━━ 14s 37ms/step - accuracy: 0.4268 - loss: 1.0020 - val_accuracy: 0.3366 - val_loss: 1.6186
Epoch 10/25
367/368 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4513 - loss: 0.8949

368/368 ━━━━━━━━━━━━━━━━━━━━ 13s 36ms/step - accuracy: 0.4512 - loss: 0.8957 - val_accuracy: 0.3673 - val_loss: 1.5748
Epoch 11/25
368/368 ━━━━━━━━━━━━━━━━━━━━ 13s 36ms/step - accuracy: 0.4642 - loss: 0.8445 - val_accuracy: 0.3611 - val_loss: 1.6294
Epoch 12/25
368/368 ━━━━━━━━━━━━━━━━━━━━ 13s 35ms/step - accuracy: 0.4984 - loss: 0.7542 - val_accuracy: 0.3879 - val_loss: 1.6467
Epoch 13/25
368/368 ━━━━━━━━━━━━━━━━━━━━ 18s 49ms/step - accuracy: 0.5084 - loss: 0.7205 - val_accuracy: 0.4254 - val_loss: 1.5858
Epoch 14/25
368/368 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.5442 - loss: 0.6598

368/368 ━━━━━━━━━━━━━━━━━━━━ 43s 117ms/step - accuracy: 0.5442 - loss: 0.6598 - val_accuracy: 0.4216 - val_loss: 1.5678
Epoch 15/25
368/368 ━━━━━━━━━━━━━━━━━━━━ 36s 97ms/step - accuracy: 0.5584 - loss: 0.6290 - val_accuracy: 0.4200 - val_loss: 1.6481
Epoch 16/25
368/368 ━━━━━━━━━━━━━━━━━━━━ 42s 114ms/step - accuracy: 0.6001 - loss: 0.5599 - val_accuracy: 0.4392 - val_loss: 1.6219
Epoch 17/25
368/368 ━━━━━━━━━━━━━━━━━━━━ 82s 114ms/step - accuracy: 0.6219 - loss: 0.5305 - val_accuracy: 0.4568 - val_loss: 1.6471
Epoch 18/25
368/368 ━━━━━━━━━━━━━━━━━━━━ 82s 113ms/step - accuracy: 0.6384 - loss: 0.4915 - val_accuracy: 0.4660 - val_loss: 1.6195
Epoch 19/25
368/368 ━━━━━━━━━━━━━━━━━━━━ 43s 116ms/step - accuracy: 0.6586 - loss: 0.4650 - val_accuracy: 0.4675 - val_loss: 1.6916
Epoch 20/25
368/368 ━━━━━━━━━━━━━━━━━━━━ 34s 91ms/step - accuracy: 0.6855 - loss: 0.4289 - val_accuracy: 0.4996 - val_loss: 1.6589
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


2026/08/03 21:27:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.




=== Neural Network Training Complete! Evaluation Phase ===

103/103 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step

Final Test Accuracy: 0.4480
Final Test Macro F1: 0.5042

Successfully logged everything to MLflow! Go check the UI!


In [37]:
print(X_train.shape)
print(X_test.shape)

print(len(tokenizer.word_index))


model.summary()

(13070, 50)
(3268, 50)
3852


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 50, 100)        │       385,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 50, 128)        │        84,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_4          │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,436,060 (5.48 MB)

 Trainable params: 478,686 (1.83 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 957,374 (3.65 MB)